#### Load the data

In [1]:
import pandas as pd
import numpy as np

X_train = pd.read_csv('../data/processed/X_train.csv')
X_val   = pd.read_csv('../data/processed/X_val.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')

y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_val   = pd.read_csv('../data/processed/y_val.csv').squeeze()
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()

print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

X_train: (192165, 11)
X_val:   (48042, 11)
X_test:  (60052, 11)


##### Initialize XGBoost model

In [2]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=3000,
    max_depth=6,
    learning_rate=0.5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
    early_stopping_rounds = 50
)


##### Train the model 

In [3]:
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

[0]	validation_0-rmse:12293.14335
[100]	validation_0-rmse:3531.66969
[200]	validation_0-rmse:3321.48159
[300]	validation_0-rmse:3217.77248
[400]	validation_0-rmse:3163.71036
[500]	validation_0-rmse:3132.78097
[600]	validation_0-rmse:3109.33447
[700]	validation_0-rmse:3095.45650
[800]	validation_0-rmse:3082.19923
[880]	validation_0-rmse:3077.48926


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [4]:
print("Training complete!")
print("Best iteration:", xgb_model.best_iteration)
print("Best score: ", xgb_model.best_score)

Training complete!
Best iteration: 830
Best score:  3077.2197637255244


#### Calculate metrics

In [5]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Get predictions
y_pred_train = xgb_model.predict(X_train)
y_pred_val   = xgb_model.predict(X_val)

def evaluate(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"=== {label} ===")
    print(f"MAE :  {mae:.2f}")
    print(f"RMSE:  {rmse:.2f}")
    print(f"R²  :  {r2:.4f}")
    print()

evaluate(y_train, y_pred_train, "TRAIN")
evaluate(y_val,   y_pred_val,   "VALIDATION")

=== TRAIN ===
MAE :  1501.04
RMSE:  2603.47
R²  :  0.9869

=== VALIDATION ===
MAE :  1727.44
RMSE:  3077.22
R²  :  0.9816



##### Initialize LightGBM model

In [6]:
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

lgb_model = LGBMRegressor(
    n_estimators=5000,
    max_depth=6,
    learning_rate=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)


##### Train the model

In [7]:

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        early_stopping(stopping_rounds=50),
        log_evaluation(period=100)
    ]
)

print("Best iteration:", lgb_model.best_iteration_)
print("Best score:    ", lgb_model.best_score_)

Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 1.65444e+07
[200]	valid_0's l2: 1.37951e+07
[300]	valid_0's l2: 1.2613e+07
[400]	valid_0's l2: 1.17799e+07
[500]	valid_0's l2: 1.11538e+07
[600]	valid_0's l2: 1.07499e+07
[700]	valid_0's l2: 1.04402e+07
[800]	valid_0's l2: 1.01776e+07
[900]	valid_0's l2: 1.00213e+07
[1000]	valid_0's l2: 9.86775e+06
[1100]	valid_0's l2: 9.73977e+06
[1200]	valid_0's l2: 9.62731e+06
[1300]	valid_0's l2: 9.53213e+06
[1400]	valid_0's l2: 9.4366e+06
[1500]	valid_0's l2: 9.3777e+06
[1600]	valid_0's l2: 9.33056e+06
[1700]	valid_0's l2: 9.2878e+06
[1800]	valid_0's l2: 9.23323e+06
[1900]	valid_0's l2: 9.19655e+06
[2000]	valid_0's l2: 9.15556e+06
[2100]	valid_0's l2: 9.10599e+06
[2200]	valid_0's l2: 9.08194e+06
[2300]	valid_0's l2: 9.04384e+06
[2400]	valid_0's l2: 9.01377e+06
[2500]	valid_0's l2: 8.99257e+06
[2600]	valid_0's l2: 8.9791e+06
[2700]	valid_0's l2: 8.96147e+06
[2800]	valid_0's l2: 8.94469e+06
[2900]	valid_0's l2: 8.92102e

#### Calculate metrics

In [8]:
y_pred_train_lgb = lgb_model.predict(X_train)
y_pred_val_lgb   = lgb_model.predict(X_val)

evaluate(y_train, y_pred_train_lgb, "LGBM TRAIN")
evaluate(y_val,   y_pred_val_lgb,   "LGBM VALIDATION")

=== LGBM TRAIN ===
MAE :  1407.31
RMSE:  2555.06
R²  :  0.9873

=== LGBM VALIDATION ===
MAE :  1603.84
RMSE:  2977.25
R²  :  0.9827



#### Intialize the function of hyperparmeter tuning

In [9]:
import optuna
from sklearn.metrics import mean_squared_error

# Silence optuna logs except warnings
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators':     3000,
        'learning_rate':    trial.suggest_float('learning_rate',   0.01, 0.3,  log=True),
        'max_depth':        trial.suggest_int(  'max_depth',       3,    10),
        'num_leaves':       trial.suggest_int(  'num_leaves',      20,   300),
        'subsample':        trial.suggest_float('subsample',       0.6,  1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree',0.6,  1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha',       0.0,  10.0),
        'reg_lambda':       trial.suggest_float('reg_lambda',      0.0,  10.0),
        'random_state':     42,
        'n_jobs':           -1,
        'verbose':          -1
    }

    model = LGBMRegressor(**params)

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[early_stopping(50), log_evaluation(0)]
    )

    y_pred = model.predict(X_val)
    rmse   = np.sqrt(mean_squared_error(y_val, y_pred))
    return rmse

print("Objective function defined successfully")

Objective function defined successfully


/home/abood-linux/miniconda3/envs/flight-price-predictor/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##### Start 50 round training in different parameters using the optuna function that we created above 

In [10]:
# Create study
study = optuna.create_study(direction='minimize')

# Run optimization
study.optimize(objective, n_trials=50, show_progress_bar=True)


  0%|          | 0/50 [00:00<?, ?it/s]

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2997]	valid_0's l2: 8.8051e+06


Best trial: 0. Best value: 2967.34:   2%|▏         | 1/50 [00:36<29:28, 36.10s/it]

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[3000]	valid_0's l2: 9.58106e+06


Best trial: 0. Best value: 2967.34:   4%|▍         | 2/50 [00:56<21:31, 26.91s/it]

...

Best trial: 43. Best value: 2931.07:  96%|█████████▌| 48/50 [24:12<01:08, 34.14s/it]

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1459]	valid_0's l2: 8.59039e+06


Best trial: 48. Best value: 2930.94:  98%|█████████▊| 49/50 [24:34<00:30, 30.68s/it]

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2278]	valid_0's l2: 8.64129e+06


Best trial: 48. Best value: 2930.94: 100%|██████████| 50/50 [25:06<00:00, 30.13s/it]


##### print best params + best RMSE

In [11]:

print("Best RMSE:  ", study.best_value)
print("Best params:", study.best_params)

Best RMSE:   2930.9375527831094
Best params: {'learning_rate': 0.06142816781183586, 'max_depth': 9, 'num_leaves': 300, 'subsample': 0.776537345864068, 'colsample_bytree': 0.7645652841781226, 'reg_alpha': 6.600295788878317, 'reg_lambda': 0.12347171561259429}


##### Back to train model in best params and for sure the giving RMSE value = above RMSE 

In [12]:
best_params = study.best_params
best_params['n_estimators']  = 5000
best_params['random_state']  = 42
best_params['n_jobs']        = -1
best_params['verbose']       = -1

tuned_lgb = LGBMRegressor(**best_params)

tuned_lgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[early_stopping(50), log_evaluation(100)]
)

print("Best iteration:", tuned_lgb.best_iteration_)

Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 1.42047e+07
[200]	valid_0's l2: 1.10693e+07
[300]	valid_0's l2: 1.00067e+07
[400]	valid_0's l2: 9.53547e+06
[500]	valid_0's l2: 9.26139e+06
[600]	valid_0's l2: 9.05721e+06
[700]	valid_0's l2: 8.89151e+06
[800]	valid_0's l2: 8.80089e+06
[900]	valid_0's l2: 8.73595e+06
[1000]	valid_0's l2: 8.694e+06
[1100]	valid_0's l2: 8.66167e+06
[1200]	valid_0's l2: 8.64322e+06
[1300]	valid_0's l2: 8.6206e+06
[1400]	valid_0's l2: 8.59505e+06
[1500]	valid_0's l2: 8.59398e+06
Early stopping, best iteration is:
[1459]	valid_0's l2: 8.59039e+06
Best iteration: 1459


##### calculate metrics for best params

In [13]:
y_pred_train_tuned = tuned_lgb.predict(X_train)
y_pred_val_tuned   = tuned_lgb.predict(X_val)

evaluate(y_train, y_pred_train_tuned, "TUNED LGBM TRAIN")
evaluate(y_val,   y_pred_val_tuned,   "TUNED LGBM VALIDATION")

=== TUNED LGBM TRAIN ===
MAE :  1287.82
RMSE:  2418.90
R²  :  0.9887

=== TUNED LGBM VALIDATION ===
MAE :  1522.16
RMSE:  2930.94
R²  :  0.9833



##### Finally test our best model on unSeen data (test set)

In [14]:
y_pred_test = tuned_lgb.predict(X_test)
evaluate(y_test, y_pred_test, "FINAL TEST SET")

=== FINAL TEST SET ===
MAE :  1515.15
RMSE:  2912.89
R²  :  0.9835



#### Save best performance model 

In [15]:
import joblib
import os

os.makedirs('../models', exist_ok=True)

joblib.dump(tuned_lgb, '../models/tuned_lgb_model.joblib')
print("Model saved successfully!")

Model saved successfully!
